# Week 4 - Statistical Model Selection & Hypothesis Development
**Course:** DATA 200 - Applied Statistical Analysis  
**Project:** EPL Match Outcome Prediction  
**Target Variable:** FTR (Full Time Result) - H, D, A

## Cell 1 - Install & Import Libraries
Install required packages and import all standard libraries used throughout this notebook.

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn -q

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Colour theme
COLOURS = {'H': '#1A5C38', 'D': '#B45309', 'A': '#8B0000'}
COLOUR_LIST = [COLOURS['H'], COLOURS['D'], COLOURS['A']]

print('All libraries loaded!')

## Cell 2 - Load Data
Load the combined EPL dataset and apply the standard FTR encoding.

In [ ]:
df = pd.read_csv('EPL_combined.csv')
df['FTR_encoded'] = df['FTR'].map({'A': 0, 'D': 1, 'H': 2})

print(f'Total rows: {len(df)}')
print(f'\nColumns ({len(df.columns)}):')
print(list(df.columns))

print('\nFTR Distribution:')
ftr_counts = df['FTR'].value_counts()
ftr_pct = df['FTR'].value_counts(normalize=True) * 100
ftr_summary = pd.DataFrame({'Count': ftr_counts, 'Percentage': ftr_pct.round(2)})
ftr_summary.index.name = 'Result'
print(ftr_summary)

## Cell 3 - Problem Statement
Explain the classification problem, class distribution, and why Multinomial Logistic Regression is the correct model choice.

In [ ]:
ftr_counts = df['FTR'].value_counts()
ftr_pct = df['FTR'].value_counts(normalize=True) * 100

print('=' * 65)
print('PROBLEM STATEMENT - EPL MATCH OUTCOME PREDICTION')
print('=' * 65)

print('\nTARGET VARIABLE: FTR (Full Time Result)')
print('  H = Home Win   -- the home team won at full time')
print('  D = Draw       -- both teams finished level')
print('  A = Away Win   -- the away team won at full time')

print('\nCLASS DISTRIBUTION:')
print('-' * 35)
for label, name in [('H', 'Home Win'), ('D', 'Draw'), ('A', 'Away Win')]:
    print(f'  {name:12s} ({label}): {ftr_counts[label]:4d} matches  ({ftr_pct[label]:.1f}%)')

print('\nWHY IS THIS A 3-CLASS CLASSIFICATION PROBLEM?')
print('-' * 46)
print('  Each match has exactly one of three mutually exclusive outcomes.')
print('  Binary logistic regression handles only 2 classes (e.g., win/loss).')
print('  Linear regression is inappropriate -- FTR is categorical, not continuous.')

print('\nWHY MULTINOMIAL LOGISTIC REGRESSION?')
print('-' * 38)
print('  1. Models P(H), P(D), P(A) simultaneously -- probabilities sum to 1')
print('  2. Produces interpretable coefficients per feature per class')
print('  3. Handles natural class imbalance without distortion')
print('  4. Outputs probabilities useful for hypothesis testing')
print('  5. Well-suited to tabular in-match statistics as predictors')
print('=' * 65)

## Cell 4 - FTR Distribution Plot
Visualise the class distribution as a pie chart and bar chart to justify the multinomial approach.

In [ ]:
ftr_counts = df['FTR'].value_counts().reindex(['H', 'D', 'A'])
ftr_pct = ftr_counts / ftr_counts.sum() * 100
labels = ['Home Win (H)', 'Draw (D)', 'Away Win (A)']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Why Multinomial? -- Our Target Has 3 Classes', fontsize=14, fontweight='bold', y=1.02)

axes[0].pie(
    ftr_counts.values, labels=labels, colors=COLOUR_LIST,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11}
)
axes[0].set_title('Proportion of Match Outcomes', fontsize=12)

bars = axes[1].bar(labels, ftr_counts.values, color=COLOUR_LIST, edgecolor='white', linewidth=0.8)
for bar, count in zip(bars, ftr_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_title('Count of Match Outcomes', fontsize=12)
axes[1].set_ylabel('Number of Matches')
axes[1].set_ylim(0, max(ftr_counts.values) * 1.15)
axes[1].tick_params(axis='x', labelsize=10)

plt.tight_layout()
plt.savefig('week4_ftr_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: week4_ftr_distribution.png')

## Cell 5 - Feature Selection
Document which features are kept, which are dropped, and why.

In [ ]:
print('=' * 65)
print('FEATURE SELECTION')
print('=' * 65)
print('\nFEATURES KEPT (10):')
print('  HST   -- Home Shots on Target   : quality of home attacking play')
print('  AST   -- Away Shots on Target   : quality of away attacking play')
print('  HC    -- Home Corners           : home team territorial pressure')
print('  AC    -- Away Corners           : away team territorial pressure')
print('  HY    -- Home Yellow Cards      : home team disciplinary record')
print('  AY    -- Away Yellow Cards      : away team disciplinary record')
print('  HR    -- Home Red Cards         : home team serious disciplinary events')
print('  AR    -- Away Red Cards         : away team serious disciplinary events')
print('  HTHG  -- Half Time Home Goals   : home team half-time performance')
print('  HTAG  -- Half Time Away Goals   : away team half-time performance')
print('\nFEATURES DROPPED (2):')
print('  HS    -- Home Shots (total)     : DROPPED -- multicollinearity with HST (r > 0.70)')
print('  AS    -- Away Shots (total)     : DROPPED -- multicollinearity with AST (r > 0.70)')
print('\nWHAT IS MULTICOLLINEARITY?')
print('  Multicollinearity occurs when two or more predictor variables are')
print('  highly correlated with each other. When HS and HST are both included,')
print('  the model cannot reliably separate their individual effects on FTR.')
print('  This inflates coefficient variance, making estimates unstable and')
print('  hypothesis tests unreliable.')
print('\nDECISION: Keep HST and AST (shot quality) over HS and AS (shot volume)')
print('  because shots on target are more directly predictive of goals.')
print('=' * 65)

## Cell 6 - Multicollinearity Visualisation
Scatter plots showing the high correlation between HS/HST and AS/AST, justifying the feature exclusion.

In [ ]:
r_hs_hst = df['HS'].corr(df['HST'])
r_as_ast = df['AS'].corr(df['AST'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Multicollinearity -- Why HS and AS Were Dropped', fontsize=13, fontweight='bold')

for ax, x_col, y_col, r_val in [
    (axes[0], 'HS', 'HST', r_hs_hst),
    (axes[1], 'AS', 'AST', r_as_ast)
]:
    for ftr_val, colour in COLOURS.items():
        mask = df['FTR'] == ftr_val
        ax.scatter(df.loc[mask, x_col], df.loc[mask, y_col],
                   color=colour, alpha=0.5, s=20, label=ftr_val)
    ax.set_xlabel(x_col, fontsize=11)
    ax.set_ylabel(y_col, fontsize=11)
    ax.set_title(f'{x_col} vs {y_col}  (r = {r_val:.3f})', fontsize=11)
    ax.legend(title='FTR', fontsize=9)

plt.tight_layout()
plt.savefig('week4_multicollinearity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Pearson r (HS vs HST): {r_hs_hst:.3f}')
print(f'Pearson r (AS vs AST): {r_as_ast:.3f}')
print('Saved: week4_multicollinearity.png')

## Cell 7 - Research Hypotheses
State all three hypotheses with supporting evidence from the dataset.

In [ ]:
hst_means = df.groupby('FTR')['HST'].mean()
htr_ftr = pd.crosstab(df['HTR'], df['FTR'], normalize='index') * 100
no_rc_away  = (df[df['HR'] == 0]['FTR'] == 'A').mean() * 100
has_rc_away = (df[df['HR'] > 0]['FTR'] == 'A').mean() * 100
ht_lead_ft_win = htr_ftr.loc['H', 'H'] if 'H' in htr_ftr.index else 0

print('=' * 65)
print('RESEARCH HYPOTHESES')
print('=' * 65)

print('\nH1: Home shots on target (HST) differs significantly across')
print('    FTR classes (H, D, A).')
print('    Test: One-Way ANOVA (Week 5)')
print('    Supporting evidence (group means):')
print(f'      Home Win  (H): HST mean = {hst_means.get("H", 0):.2f}')
print(f'      Draw      (D): HST mean = {hst_means.get("D", 0):.2f}')
print(f'      Away Win  (A): HST mean = {hst_means.get("A", 0):.2f}')

print('\nH2: The half-time result (HTHG/HTAG) is a significant predictor')
print('    of full-time result (FTR).')
print('    Test: Logistic regression coefficient (Week 5)')
print(f'    Supporting evidence: {ht_lead_ft_win:.1f}% of HT leaders win at full time')

print('\nH3: Receiving a red card significantly shifts match outcome')
print('    probabilities.')
print('    Test: Logistic regression coefficient + proportion analysis (Week 5)')
print(f'    Away win rate -- no home red card : {no_rc_away:.1f}%')
print(f'    Away win rate -- home has red card: {has_rc_away:.1f}%')
print('=' * 65)

## Cell 8 - Hypothesis Visualisations
Three-panel figure providing visual evidence for each hypothesis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Hypothesis Evidence -- Week 4 Preview', fontsize=13, fontweight='bold')

# H1: Mean HST by FTR
hst_means = df.groupby('FTR')['HST'].mean().reindex(['H', 'D', 'A'])
bars = axes[0].bar(['Home Win', 'Draw', 'Away Win'], hst_means.values,
                   color=COLOUR_LIST, edgecolor='white')
for bar, val in zip(bars, hst_means.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('H1: Mean HST by Match Outcome', fontsize=11)
axes[0].set_ylabel('Mean Shots on Target (Home)')
axes[0].set_ylim(0, hst_means.max() * 1.2)

# H2: HTR to FTR heatmap
htr_ftr = pd.crosstab(df['HTR'], df['FTR'], normalize='index') * 100
htr_ftr = htr_ftr.reindex(index=['H', 'D', 'A'], columns=['H', 'D', 'A'])
sns.heatmap(htr_ftr, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=axes[1], cbar_kws={'label': '%'},
            xticklabels=['Home Win', 'Draw', 'Away Win'],
            yticklabels=['HT Home Lead', 'HT Draw', 'HT Away Lead'])
axes[1].set_title('H2: HTR to FTR Transition (%)', fontsize=11)
axes[1].set_xlabel('Full Time Result')
axes[1].set_ylabel('Half Time Result')

# H3: Grouped bar -- both directions
no_rc_away  = (df[df['HR'] == 0]['FTR'] == 'A').mean() * 100
has_rc_away = (df[df['HR'] > 0]['FTR'] == 'A').mean() * 100
no_rc_home  = (df[df['AR'] == 0]['FTR'] == 'H').mean() * 100
has_rc_home = (df[df['AR'] > 0]['FTR'] == 'H').mean() * 100

x = np.arange(2)
w = 0.35
axes[2].bar(x - w/2, [no_rc_away, no_rc_home],  w, label='No Red Card', color='#4A90D9', alpha=0.85)
axes[2].bar(x + w/2, [has_rc_away, has_rc_home], w, label='Red Card',    color='#8B0000', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(['Away Win\n(Home RC)', 'Home Win\n(Away RC)'], fontsize=10)
axes[2].set_ylabel('Win Rate (%)')
axes[2].set_title('H3: Red Card Effect -- Both Directions', fontsize=11)
axes[2].legend(fontsize=9)
axes[2].set_ylim(0, 100)
for i, (no, has) in enumerate([(no_rc_away, has_rc_away), (no_rc_home, has_rc_home)]):
    axes[2].text(i - w/2, no  + 1, f'{no:.1f}%',  ha='center', fontsize=9)
    axes[2].text(i + w/2, has + 1, f'{has:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('week4_hypotheses.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: week4_hypotheses.png')

## Cell 9 - Why ANOVA?
Explain the rationale for using One-Way ANOVA as a pre-test before logistic regression.

In [ ]:
print('=' * 65)
print('WHY ONE-WAY ANOVA?')
print('=' * 65)
print('\nWHAT ANOVA TESTS:')
print('  One-Way ANOVA tests whether the means of a continuous variable')
print('  differ significantly across 3 or more groups.')
print('  Here: does mean HST differ between Home Win, Draw, Away Win?')
print('\nWHY ANOVA BEFORE LOGISTIC REGRESSION:')
print('  ANOVA is a univariate pre-screening step. Before building a')
print('  multi-feature model, we confirm that individual features show')
print('  meaningful group separation. Features that pass ANOVA (p < 0.05)')
print('  are statistically justified as predictors in the logistic model.')
print('\nFEATURES TO BE TESTED IN WEEK 5:')
print('  HST, AST, HC, AC')
print('\nINTERPRETING p < 0.05:')
print('  Reject the null hypothesis that all group means are equal.')
print('  The feature differs significantly across match outcomes.')
print('  p < 0.001 is marked as *** (very strong evidence).')
print('=' * 65)

## Cell 10 - Week 4 Summary

| Decision | Choice | Rationale |
|---|---|---|
| Model | Multinomial Logistic Regression | 3-class categorical target; interpretable coefficients |
| Solver | lbfgs, max_iter=1000 | Stable for multi-class; handles moderate feature count |
| FTR Encoding | A=0, D=1, H=2 | Ordinal-style encoding for sklearn compatibility |
| Features Kept | HST, AST, HC, AC, HY, AY, HR, AR, HTHG, HTAG (10) | Low multicollinearity; directly predictive |
| Features Dropped | HS, AS | Pearson r > 0.70 with HST/AST -- multicollinearity |
| Train/Test Split | 80/20, random_state=42, stratify=y | Reproducible; preserves class proportions |
| Scaling | StandardScaler (fit on train only) | Prevents data leakage; required for lbfgs |
| Class Imbalance | Natural imbalance retained | Reflects real-world football distributions |
| H1 | HST differs across FTR groups | One-Way ANOVA (Week 5) |
| H2 | HTHG/HTAG predicts FTR | Logistic regression coefficient (Week 5) |
| H3 | Red cards shift outcome probabilities | Coefficient + grouped proportion analysis (Week 5) |
| Pre-test | One-Way ANOVA for HST, AST, HC, AC | Validates feature group separation before modelling |